In [6]:
import numpy as np
import nibabel as nib
import trimesh

from scipy.ndimage import binary_closing, binary_fill_holes, distance_transform_edt
from skimage.measure import marching_cubes
from trimesh.smoothing import filter_taubin


# Settings
LABEL_FILE = "Brain_label.nii"
CLOSING_RADIUS = 2
TAUBIN_ITERATIONS = 20
TAUBIN_LAMBDA = 0.5
TAUBIN_MU = -0.53
SAMPLE_OFFSET = 0.75
MIN_LABEL = 1
MAX_LABEL = 86

OUTPUT_OBJ_COLOR = f"brain_segmented_colored_{CLOSING_RADIUS:02d}.obj"
OUTPUT_OBJ_GREY = f"brain_segmented_grey_{CLOSING_RADIUS:02d}.obj"
OUTPUT_COLOR_CSV = f"brain_color_{CLOSING_RADIUS:02d}.csv"


# Anatomical label colors.
COLORS = {
    1:  (237, 216, 166),
    2:  (213, 91, 43),
    3:  (213, 91, 43),
    4:  (244, 201, 68),
    5:  (244, 201, 68),
    6:  (178, 210, 67),
    7:  (178, 210, 67),
    8:  (108, 186, 57),
    9:  (108, 186, 57),
    10: (186, 235, 189),
    11: (186, 235, 189),
    12: (83, 149, 186),
    13: (83, 149, 186),
    14: (56, 70, 132),
    15: (56, 70, 132),
    16: (172, 217, 111),
    17: (172, 217, 111),
    18: (209, 134, 193),
    19: (209, 134, 193),
    20: (203, 74, 150),
    21: (203, 74, 150),
    22: (174, 117, 72),
    23: (174, 117, 72),
    24: (243, 228, 80),
    25: (243, 228, 80),
    26: (241, 250, 215),
    27: (241, 250, 215),
    28: (126, 199, 89),
    29: (126, 199, 89),
    30: (187, 150, 147),
    31: (187, 150, 147),
    32: (248, 213, 200),
    33: (248, 213, 200),
    34: (201, 60, 45),
    35: (201, 60, 45),
    36: (140, 196, 59),
    37: (140, 196, 59),
    38: (204, 58, 50),
    39: (204, 58, 50),
    40: (192, 235, 230),
    41: (192, 235, 230),
    42: (168, 221, 229),
    43: (168, 221, 229),
    44: (60, 58, 153),
    45: (60, 58, 153),
    46: (113, 75, 174),
    47: (113, 75, 174),
    48: (245, 220, 74),
    49: (245, 220, 74),
    50: (123, 191, 57),
    51: (123, 191, 57),
    52: (233, 134, 173),
    53: (233, 134, 173),
    54: (229, 128, 61),
    55: (229, 128, 61),
    56: (246, 224, 100),
    57: (246, 224, 100),
    58: (90, 132, 53),
    59: (90, 132, 53),
    60: (191, 220, 96),
    61: (191, 220, 96),
    62: (237, 242, 251),
    63: (237, 242, 251),
    64: (203, 226, 237),
    65: (203, 226, 237),
    66: (146, 133, 204),
    67: (146, 133, 204),
    68: (185, 231, 228),
    69: (185, 231, 228),
    70: (249, 237, 139),
    71: (247, 205, 84),
    72: (148, 65, 176),
    73: (251, 245, 250),
    74: (119, 38, 29),
    75: (150, 198, 65),
    76: (219, 95, 74),
    77: (255, 255, 255),
    78: (213, 115, 44),
    79: (213, 115, 44),
    80: (176, 213, 233),
    81: (134, 88, 185),
    82: (134, 88, 185),
    83: (206, 50, 70),
    84: (206, 50, 70),
    85: (134, 88, 185),
    86: (134, 88, 185),
}


def sample_nearest(volume, points, default=0):
    """Sample a 3-D array at floating point voxel coordinates."""
    ijk = np.rint(points).astype(np.intp)
    inside = np.all((ijk >= 0) & (ijk < np.asarray(volume.shape)), axis=1)
    values = np.full(len(points), default, dtype=volume.dtype)
    values[inside] = volume[tuple(ijk[inside].T)]
    return values


# 1. Load labels and prepare a cleaned binary mask.
nii = nib.load(LABEL_FILE)
labels = np.rint(nii.get_fdata()).astype(np.int16)
if labels.ndim != 3:
    raise ValueError(f"Expected a 3-D label volume, got shape {labels.shape}")

anatomical_mask = (labels >= MIN_LABEL) & (labels <= MAX_LABEL)
if not anatomical_mask.any():
    raise ValueError("No anatomical labels were found in the label volume.")

mask = anatomical_mask.copy()
if CLOSING_RADIUS > 0:
    width = 2 * CLOSING_RADIUS + 1
    mask = binary_closing(mask, structure=np.ones((width,) * 3, dtype=bool))
mask = binary_fill_holes(mask, structure=np.ones((3, 3, 3), dtype=bool))

# Nearest original anatomical voxel supplies labels for mask-created surface points.
nearest_indices = distance_transform_edt(~anatomical_mask, return_indices=True)[1]

# 2. Extract the surface. Degenerate triangles are filtered by marching cubes.
verts_voxel, faces, normals_voxel, _ = marching_cubes(
    mask.astype(np.uint8), level=0.5, allow_degenerate=False
)

# 3. Taubin smooth in world coordinates, respecting voxel spacing.
verts_world = nib.affines.apply_affine(nii.affine, verts_voxel)
if TAUBIN_ITERATIONS > 0:
    smooth_mesh = trimesh.Trimesh(vertices=verts_world, faces=faces, process=False)
    filter_taubin(
        smooth_mesh,
        lamb=TAUBIN_LAMBDA,
        nu=TAUBIN_MU,
        iterations=TAUBIN_ITERATIONS,
    )
    verts_world = smooth_mesh.vertices.copy()
verts_voxel = nib.affines.apply_affine(np.linalg.inv(nii.affine), verts_world)

# 4. Sample one anatomical label and color at every vertex.
# Marching-cubes normals are used directly; choose the side inside the mask.
side_a = verts_voxel + SAMPLE_OFFSET * normals_voxel
side_b = verts_voxel - SAMPLE_OFFSET * normals_voxel
inside_a = sample_nearest(mask, side_a, default=False).astype(bool)
inside_b = sample_nearest(mask, side_b, default=False).astype(bool)
sample_points = verts_voxel.copy()
sample_points[inside_a & ~inside_b] = side_a[inside_a & ~inside_b]
sample_points[inside_b & ~inside_a] = side_b[inside_b & ~inside_a]

vertex_labels = sample_nearest(labels, sample_points, default=0).astype(np.int16)
color_by_label = {label: (*rgb, 255) for label, rgb in COLORS.items()}
# Any surface sample without a mapped anatomical color inherits the nearest
# original anatomical label, including labels created by mask closing/filling.
missing = np.flatnonzero(~np.isin(vertex_labels, list(color_by_label)))
if len(missing):
    ijk = np.rint(sample_points[missing]).astype(np.intp)
    ijk = np.clip(ijk, 0, np.asarray(labels.shape) - 1)
    nearest = nearest_indices[:, ijk[:, 0], ijk[:, 1], ijk[:, 2]]
    vertex_labels[missing] = labels[tuple(nearest)]

vertex_colors = np.asarray(
    [color_by_label[int(label)] for label in vertex_labels],
    dtype=np.uint8,
)
del nearest_indices

# 5. Export both meshes and a CSV aligned with the colored OBJ's vertices.
color_mesh = trimesh.Trimesh(vertices=verts_world, faces=faces, process=False)
color_mesh.visual.vertex_colors = vertex_colors
color_mesh.export(OUTPUT_OBJ_COLOR, include_normals=False)

grey_mesh = trimesh.Trimesh(vertices=verts_world, faces=faces, process=False)
grey_mesh.export(OUTPUT_OBJ_GREY, include_normals=False)

color_table = np.column_stack((
    np.arange(len(verts_world), dtype=np.int64),
    vertex_labels.astype(np.int64),
    vertex_colors.astype(np.int64),
))
np.savetxt(
    OUTPUT_COLOR_CSV,
    color_table,
    delimiter=",",
    fmt="%d",
    header="vertex_index,label,red,green,blue,alpha",
    comments="",
)

print(f"Saved {OUTPUT_OBJ_GREY}")
print(f"Saved {OUTPUT_OBJ_COLOR}")
print(f"Saved {OUTPUT_COLOR_CSV}")
print(f"Mesh: {len(verts_world):,} vertices, {len(faces):,} faces")
print(f"Vertices assigned nearest anatomical labels: {len(missing):,}")



Saved brain_segmented_grey_02.obj
Saved brain_segmented_colored_02.obj
Saved brain_color_02.csv
Mesh: 102,728 vertices, 205,472 faces
Vertices assigned nearest anatomical labels: 23,432
